In [38]:
import pandas as pd
import tqdm
import requests
import json

from utils import MetricHelper

import utils

In [ ]:
df = pd.read_csv('Data_FailureFixExplanation.csv')
ref_df = pd.read_csv('Root_Cause_Ref.csv')

In [31]:
# API provided by running llm using Ollama locally
model_deepseek_llm = "deepseek-llm:7b"
model_llama = "llama3.2:3b"
data = {
    "model": model_deepseek_llm,
    "prompt": "",
    "stream": False
}

prompt_task_professional = "###Task:\nYou are a professional software developer. You are given a number of explanations describing the root cause of a software failure. Based on the given explanations, write a single explanation that contains all the information required to understand the root cause of the bug. The explanation should be succinct and without redundant information"

prompt_input = "### Input:\n\nHere are the failure explanations:\n\n"

prompt_output = "### Output:\nFormat your response in valid JSON format with a single field 'explanation' of type string containing your generated explanation."

In [33]:
methods = df['File'].unique().tolist()

ref_c = {method: expl for method, expl in zip(ref_df['bug'].to_list(), ref_df['description_c'].to_list())}
ref_d = {method: expl for method, expl in zip(ref_df['bug'].to_list(), ref_df['description_d'].to_list())}

In [22]:
def generate_explanation(explanations, data, prompt_task, prompt_input='', prompt_output=''):
    url = "http://localhost:11434/api/generate"
    headers = {
        "Content-Type": "application/json"
    }
    input = '\n\n' + '\n\n'.join(["'''\n" + expl + "\n'''" for expl in explanations]) + '\n\n'

    prompt = prompt_task + prompt_input + input + prompt_output
    data['prompt'] = prompt
    response = requests.post(url, headers=headers, data=json.dumps(data))
    return json.loads(response.text)['response']

In [17]:
schema = {
    "$schema": "ase-schema",
    "title": "Explanation",
    "description": "A failure explanation",
    "type": "object",
    "properties": {
        "explanation": {
            "description": "The generated explanation",
            "type": "string"
        }
    },
    "required": ["explanation"]
}
data_json_default = {
    "model": model_deepseek_llm,
    "prompt": "",
    "stream": False,
    "format": "json"
}
data_json = {
    "model": model_deepseek_llm,
    "prompt": "",
    "stream": False,
    "format": schema
}

In [35]:
generated_explanations = {}
for method in tqdm.tqdm(methods):
    method_explanations = df[(df['File'] == method)]['Explanation'].to_list()
    generated_explanations[method] = generate_explanation(method_explanations, data_json, prompt_task_professional, prompt_input, prompt_output)

100%|██████████| 8/8 [00:26<00:00,  3.32s/it]


In [37]:
generated_explanations.items()

dict_items([('HIT01_8', '{\n    "explanation": "The issue lies on line 279 of the code, where it checks if minutesOffset is within the range of -59 to 59. However, since minutesOffset can be a negative value, the check should instead include values less than -59."}'), ('HIT02_24', '{\n   "explanation": "The Color function does not accept negative numbers, which results from passing -0.5 to the getPaint method."\n}'), ('HIT03_6', '{\n"explanation": "The code is likely causing the StringIndexOutOfBoundsException error due to the variable pos being incremented beyond the size of variable input within the while loop, which may cause pos index going negative or out of bounds."\n}'), ('HIT04_7', '{\n"explanation": "The root cause of the software failure can be attributed to several factors including unintended changes to the variable e within the program, incorrect usage of data parameters in the source code and issues related to the calculation of MaxMiddleIndex."\n}'), ('HIT05_35', '{\n   

In [ ]:
def calculate_metrics(explanations):
    expl_metrics = {}
    for method, expl in explanations.items():
        method_metrics = {}
        ref_expl_c = ref_c[method]
        ref_expl_d = ref_d[method]
        method_metrics['bleu_c'] = MetricHelper.calculateBleuScore(ref_expl_c, expl)
        method_metrics['bleu_d'] = MetricHelper.calculateBleuScore(ref_expl_d, expl)
        expl_metrics[method] = method_metrics
    return expl_metrics